<a href="https://colab.research.google.com/github/AquilaDream/PROJETOS_PESSOAIS_PY/blob/main/prestacao_servico_vemax_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas openpyxl


In [2]:
import pandas as pd
from datetime import datetime, time

# =========================
# CONSTANTES DO TERMO
# =========================
V_DESLOC = 151.50
V_DESLOC_EXTRA = 180.00
V_SERVICO = 302.50
V_SERVICO_EXTRA = 360.00
V_KM = 2.92
V_REFEICAO = 50.00

HOR_INI = time(7, 0)
HOR_FIM = time(17, 0)
ALMOCO = 1  # 1 hora fixa

# =========================
# FUNÇÕES
# =========================
def horas(inicio, fim):
    return (fim - inicio).total_seconds() / 3600

def horas_extra(inicio, fim):
    extra = 0
    if inicio.time() < HOR_INI:
        extra += horas(inicio, datetime.combine(inicio.date(), HOR_INI))
    if fim.time() > HOR_FIM:
        extra += horas(datetime.combine(fim.date(), HOR_FIM), fim)
    return extra

# =========================
# ENTRADA DE DADOS
# =========================
cliente = input("Nome do cliente: ")
qtd_dias = int(input("Quantidade de dias trabalhados: "))

dados = []

for i in range(qtd_dias):
    print(f"\n--- DIA {i+1} ---")
    data = input("Data (DD/MM/AAAA): ")

    ida_ini = input("Início deslocamento IDA (HH:MM): ")
    ida_fim = input("Fim deslocamento IDA (HH:MM): ")

    serv_ini = input("Início serviço (HH:MM): ")
    serv_fim = input("Fim serviço (HH:MM): ")

    volta_ini = input("Início deslocamento VOLTA (HH:MM): ")
    volta_fim = input("Fim deslocamento VOLTA (HH:MM): ")

    data_dt = datetime.strptime(data, "%d/%m/%Y").date()

    ida_ini = datetime.combine(data_dt, datetime.strptime(ida_ini, "%H:%M").time())
    ida_fim = datetime.combine(data_dt, datetime.strptime(ida_fim, "%H:%M").time())
    serv_ini = datetime.combine(data_dt, datetime.strptime(serv_ini, "%H:%M").time())
    serv_fim = datetime.combine(data_dt, datetime.strptime(serv_fim, "%H:%M").time())
    volta_ini = datetime.combine(data_dt, datetime.strptime(volta_ini, "%H:%M").time())
    volta_fim = datetime.combine(data_dt, datetime.strptime(volta_fim, "%H:%M").time())

    h_ida = horas(ida_ini, ida_fim)
    h_volta = horas(volta_ini, volta_fim)
    h_desloc = h_ida + h_volta
    h_desloc_extra = horas_extra(ida_ini, ida_fim) + horas_extra(volta_ini, volta_fim)

    h_serv_bruto = horas(serv_ini, serv_fim) - ALMOCO
    h_serv_extra = max(0, h_serv_bruto - 9) + horas_extra(serv_ini, serv_fim)
    h_serv_normal = max(0, h_serv_bruto - h_serv_extra)

    dados.append({
        "DATA": data,
        "H_DESLOC": h_desloc,
        "H_DESLOC_EXTRA": h_desloc_extra,
        "H_SERVICO": h_serv_normal,
        "H_SERVICO_EXTRA": h_serv_extra
    })

# =========================
# DESPESAS
# =========================
km = float(input("\nKM rodado: "))
ped_qtd = int(input("Quantidade de pedágios: "))
ped_valor = float(input("Valor por pedágio: ").replace(',', '.'))
ref_qtd = int(input("Quantidade de refeições: "))
desp_qtd = int(input("Quantidade de despesas extras: "))
desp_valor = float(input("Valor unitário despesa extra: ").replace(',', '.'))

garantia = input("Está na garantia? (SIM/NAO): ").upper()
obr = input("Observações: ")

# =========================
# TOTALIZAÇÃO
# =========================
df = pd.DataFrame(dados)

total_desloc = df["H_DESLOC"].sum() * V_DESLOC
total_desloc_extra = df["H_DESLOC_EXTRA"].sum() * V_DESLOC_EXTRA
total_serv = df["H_SERVICO"].sum() * V_SERVICO
total_serv_extra = df["H_SERVICO_EXTRA"].sum() * V_SERVICO_EXTRA

total_km = km * V_KM
total_ped = ped_qtd * ped_valor
total_ref = ref_qtd * V_REFEICAO
total_desp = desp_qtd * desp_valor

subtotal = (
    total_desloc +
    total_desloc_extra +
    total_serv +
    total_serv_extra +
    total_km +
    total_ped +
    total_ref +
    total_desp
)

desconto = 0
if garantia == "SIM":
    desconto = total_serv + total_serv_extra

total_pagar = subtotal - desconto

# =========================
# SAÍDA FINAL
# =========================
resumo = pd.DataFrame([{
    "CLIENTE": cliente,
    "TOTAL_DESLOC": round(total_desloc, 2),
    "TOTAL_DESLOC_EXTRA": round(total_desloc_extra, 2),
    "TOTAL_SERVICO": round(total_serv, 2),
    "TOTAL_SERVICO_EXTRA": round(total_serv_extra, 2),
    "KM": round(total_km, 2),
    "PEDAGIO": round(total_ped, 2),
    "REFEICAO": round(total_ref, 2),
    "DESP_EXTRA": round(total_desp, 2),
    "DESCONTO": round(desconto, 2),
    "TOTAL_A_PAGAR": round(total_pagar, 2),
    "OBS": obr
}])

with pd.ExcelWriter("Prestacao_Servico_FINAL.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="DETALHAMENTO_DIARIO")
    resumo.to_excel(writer, index=False, sheet_name="RESUMO")

KeyboardInterrupt: Interrupted by user